In [32]:
def avaliar_roteador(eval_set, orchestrator):
    acertos = 0
    for item in eval_set:
        categoria = orchestrator.router_chain.invoke({"pergunta": item["pergunta"]}).strip().upper()
        if item["categoria_esperada"] in categoria:
            acertos += 1
        else:
            print(f"ERRO: '{item['pergunta']}' → esperado {item['categoria_esperada']}, obteve {categoria}")
    return acertos / len(eval_set)

acuracia_roteador = avaliar_roteador(EVAL_SET, orchestrator)
print(f"Acurácia do roteador: {acuracia_roteador:.1%}")

c:\Users\3\Downloads\utilities-copilot\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\3\Downloads\utilities-copilot\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\3\Downloads\utilities-copilot\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\3\Downloads\utilities-copilot\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling pa

Acurácia do roteador: 100.0%


In [25]:
def avaliar_retrieval(eval_set, buscador, k=5):
    acertos_topk = 0
    total = 0
    for item in eval_set:
        if "modulo_esperado" not in item:
            continue
        total += 1
        docs = buscador.invoke(item["pergunta"])[:k]
        encontrou = any(
            doc.metadata.get("modulo") == item["modulo_esperado"]
            for doc in docs
        )
        if encontrou:
            acertos_topk += 1
        else:
            print(f"NÃO ENCONTRADO: '{item['pergunta']}'")
            print(f"  Esperado: {item['modulo_esperado']}")
            print(f"  Retornou: {[doc.metadata.get('modulo') for doc in docs]}\n")
    return acertos_topk / total if total else 0

hit_rate_retrieval = avaliar_retrieval(EVAL_SET, buscador, k=5)
print(f"\nHit Rate@5: {hit_rate_retrieval:.1%}")

NÃO ENCONTRADO: 'O que caracteriza uma interrupção de curta duração no fornecimento de energia?'
  Esperado: Módulo 8
  Retornou: ['Módulo 1', 'Módulo 6', 'Módulo 1', 'Módulo 6', 'Módulo 1']


Hit Rate@5: 85.7%


In [34]:
docs_teste = buscador.invoke("Quais são as regras de aterramento exigidas para unidades consumidoras?")
for doc in docs_teste[:5]:
    print(doc.metadata.get("modulo"), "-", doc.page_content[:150])

Módulo 3 - aterramento temporário do equipamento ou instalação no qual se executará o serviço;
chaves de manobra e conjuntos de aterramento;
tensões de toque e d
Módulo 4 - 62. As cargas interruptíveis por contrato devem ser as primeiras indicadas para corte.

63. A Priorização de Alimentadores por Subestação – PAS deve a
Módulo 4 - 48. Os consumidores têm as seguintes atribuições no controle de carga:

manter atualizado seu cadastro junto à distribuidora para receber comunicações
Módulo 3 - (4) Cabe à distribuidora definir no estudo técnico o tempo de reconexão, baseado em normas técnicas
próprias e da ABNT. (Incluído pela REN ANEEL 1.059
Módulo 4 - Diretrizes para priorização de cargas

61. A distribuidora deve definir critérios para classificação e priorização das unidades consumidoras, para
fin


In [35]:
with open(ROOT_DIR / "src" / "agents" / "regulatory_agent.py", "r", encoding="utf-8") as f:
    conteudo = f.read()
print(conteudo)

import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ROOT_DIR = Path(__file__).resolve().parent.parent.parent
sys.path.append(str(ROOT_DIR))
load_dotenv(ROOT_DIR / ".env")

from src.reranking.rerank import Reranker

QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_NAME = "prodist_normativas"

def formatar_docs(docs):
    texto_formatado = []

    for i, doc in enumerate(docs, 1):
        documento = doc.metadata.get("documentos", "Documento não identificado")
        modulo = doc.metadata.get("modulo", "Módulo não identificado")
        pagina = doc.metadata.get("pagina", "Página não identificada")

        texto_formatado.append(
